In [ ]:
import os
import re
import json
from datetime import datetime
from typing import Optional, List, Callable, Any, Tuple, Dict, Set
from abc import abstractmethod, ABC
import random
import numpy as np
import pandas as pd
import copy
import pickle
import itertools
import string



In [ ]:
from dataset_utils.paths import get_dataset_dir
from dataset_utils.utils import file_handle, grouping
from dataset_utils.utils.tree import Node

dataset_dir = get_dataset_dir()
generated_datasets_dir = dataset_dir / "datasets"

random.seed(420)
np.random.seed(420)


# Dataset Creation 

In [ ]:
from dataset_utils.reader import ADIQDataset
from dataset_utils.creation import save_dataset
from dataset_utils.question import Question
from dataset_utils.outputs import to_basic_prompt

ds = ADIQDataset(generated_datasets_dir / "DiagnosticIQ")

In [ ]:
OPTION_STRING = string.ascii_uppercase

def convert_to_complex(
        q:Question, 
        choices:Set[str],
        rule_info:Dict[str,Any],
        rule_similarity:Dict[str,List[str]],
        all_rule_set:Dict[str,Dict[str,Any]],
        uniq_obs:Dict[str,str],
        num_options = 10,
        n_least_sim_rules = 35
        ) -> None:
    
    q.subject = "diagnosis_sense_complex_question_analysis"
    
    if q.question_type == "positive":
        sel_rules = rule_similarity[rule_info["id"]][:n_least_sim_rules]

        _obs = set()
        for r in sel_rules:
            r = all_rule_set[r]["display_text"]["observations"]
            _obs = _obs.union(set([uniq_obs[x] for x in r]))

        convert_positive_question(q, _obs, num_options, rule_info, rule_similarity)
    else:
        convert_negative_question(q, rule_info, num_options)


def convert_positive_question(
        q:Question, choices:Set[str], num_options:int, 
        rule_info:Dict[str,Any], rule_similarity:Dict[str,List[str]]
        ) -> None:
    
    
    options_in_question = set(q.options)
    available_options = choices - options_in_question - set(rule_info["display_text"]["observations"])

    if num_options-len(options_in_question)>0:
        new_options = random.sample(list(available_options), num_options - len(options_in_question))
        cor = np.array(q.options)[np.array(q.correct)][0]
        new_options.extend(list(options_in_question - {cor}))
        
        all_options = [cor] + new_options
        ind = list(range(len(all_options)))
        random.shuffle(ind)

        q.options = [all_options[x] for x in ind]
        q.correct = (np.array(ind) == 0).tolist()
        q.option_ids = [OPTION_STRING[x] for x in range(len(ind))]


def convert_negative_question(q:Question, rule_info:Dict[str,Any], num_options:int) -> None:
    options_in_question = set(q.options)
    available_options =set(rule_info["display_text"]["observations"]) - options_in_question

    if num_options-len(options_in_question)>0 and len(available_options)>0:
        if len(available_options)> num_options-len(options_in_question)>0 :
            new_options = random.sample(list(available_options), num_options - len(options_in_question))
        else:
            new_options = list(available_options)

        cor = np.array(q.options)[np.array(q.correct)][0]
        new_options.extend(list(options_in_question - {cor}))
        
        all_options = [cor] + new_options
        ind = list(range(len(all_options)))
        random.shuffle(ind)

        q.options = [all_options[x] for x in ind]
        q.correct = (np.array(ind) == 0).tolist()
        q.option_ids = [OPTION_STRING[x] for x in range(len(ind))]


In [ ]:
# options augmentation
cmplx_ques:List[Dict[str,Any]] = []

for q in ds.questions:
    convert_to_complex(
        q,
        set(ds.unique_observations.values()),
        [x for x in ds.rule_info["rule_set"] if x["#n"] == q.rule_id][0],
        ds.rule_id_similarity_map,
        {k["id"]:k for k in ds.rule_info["rule_set"]},
        ds.unique_observations
        )
    cmplx_ques.append(
        q.to_dict()
    )


ds_complex = copy.deepcopy(ds.__dict__)
ds_complex.pop("data")
ds_complex["questions"] = cmplx_ques

In [ ]:
save_dataset(
    ds_complex,
    "DiagnosticIQPro",
    generated_datasets_dir
    )

In [ ]:
ds = ADIQDataset(generated_datasets_dir / "DiagnosticIQPro")

In [ ]:
question_template = """
## Asset Description:
{asset_type}: {asset_description}

## Conditions:
{conditions}

## How long the conditions were met:
{temporal_condition}

{question_prompt}
{options}
"""

In [ ]:
print(to_basic_prompt(ds.questions[200],question_template, ds.asset_descriptions))

In [ ]:
ds.questions[2000].rule_id, ds.questions[2000].correct

In [ ]:
len(ds.questions)

In [ ]:
opt_comp = {}
for q in ds.questions:
    try:
        opt_comp[len(q.options)] += 1
    except KeyError as er:
        opt_comp[len(q.options)] = 1

In [ ]:
opt_comp